# Construct quadratic program

In [1]:
import json
import numpy as np
import networkx as nx

from itertools import combinations
from scipy.optimize import minimize
from pathlib import Path
from tqdm import tqdm

from qiskit import QuantumCircuit
from qiskit.primitives import (
    BackendSampler,
    BackendSamplerV2,
    BackendEstimator,
    BackendEstimatorV2,
    StatevectorEstimator,
    Estimator,
)
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import EfficientSU2, TwoLocal
from qiskit_optimization.applications import Maxcut
from qiskit_optimization.algorithms import MinimumEigenOptimizer, GurobiOptimizer
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_algorithms import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import NELDER_MEAD, COBYLA, L_BFGS_B
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import (
    Sampler as AerSampler,
    SamplerV2 as AerSamplerV2,
    Estimator as AerEstimator,
    EstimatorV2 as AerEstimatorV2,
)

from lcc import LCCVQE
from lcc.knapsack import Knapsack

sc140_data_dir = Path('data/140')

## Generate Max-cut instances

In [3]:
n = 10
G = nx.random_regular_graph(2, n, seed=0)
max_cut = Maxcut(G)
qp = max_cut.to_quadratic_program()
# print(qp.prettyprint())

## Load Knapsack instances

In [3]:
knapsack = Knapsack('data/90/data_2.json')
qp = knapsack.quadratic_program
conv = QuadraticProgramToQubo()
qubo = conv.convert(qp)

## Load Supply Chain instances

In [2]:
from lcc.supply_chain import construct_qiskit_quadratic_program

with open(sc140_data_dir/'data_0.json') as f:
    params = json.load(f)

qp = construct_qiskit_quadratic_program(params)
# print(qp.prettyprint())

## Exact solution

In [ ]:
exact_solver = MinimumEigenOptimizer(NumPyMinimumEigensolver())
exact_result = exact_solver.solve(qp)
print(exact_result.prettyprint())

In [3]:
gurobi_result = GurobiOptimizer().solve(qp)
true_obj = gurobi_result.fval
print('Gurobi cost:', true_obj)

Restricted license - for non-production use only - expires 2025-11-24
Gurobi cost: 65.0


# Solve using LCC

In [5]:
estimator = StatevectorEstimator()
# estimator = None
optimizer = COBYLA()

solver = LCCVQE(
    qp,
    estimator=estimator,
    optimizer=optimizer,
)

initial_params = solver.generate_random_params()
print('Initial params:', initial_params.view())
solver.compute_energy(initial_params)

Initial params: [5.01800103 4.7867131  0.91471103 3.55666011 5.01461372 0.62181845
 0.6629095  3.69498424 1.68289382 2.69463692 0.71817197 0.98035058
 1.34179051 4.9929076  5.50554066 3.56482003 1.48746878 1.80030026
 3.91769199 1.78490477 1.34514728 0.55960547 1.18271573 1.36284793
 1.00823927 4.96911364 5.55554721 5.31015976 5.41063659 2.80107049
 3.42792303 1.40994359 2.6199496  4.90449636]


-3437.456249266422

In [8]:
# estimator = StatevectorEstimator()
estimator = None
optimizer = COBYLA()

n_trials = 30
shots = int(1e5)
costs = []
solutions = []
feasibles = []

for _ in tqdm(range(n_trials)):
    solver = LCCVQE(
        qp,
        shots=shots,
        estimator=estimator,
        optimizer=optimizer,
    )

    result = solver.solve(initial_point=None, run_sampler=True)
    expectation = -(result.fun + solver.offset)
    feasible = qp.is_feasible(solver.optimal_solution)

    costs.append(expectation)
    solutions.append(solver.optimal_solution)
    feasibles.append(feasible)

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [30:06:44<00:00, 3613.49s/it]   


In [11]:
costs

[-7577343.0224609375,
 -4822348.380859375,
 -6338405.442626953,
 -6196330.437744141,
 -7529738.614013672,
 -6835613.7314453125,
 -7666255.9072265625,
 -8516909.266601562,
 -6319680.1142578125,
 -8717818.86694336,
 -4401338.521484375,
 -6404086.554931641,
 -5695169.389892578,
 -4849415.174072266,
 -5772621.152587891,
 -7458005.282714844,
 -10470649.928222656,
 -7141549.844238281,
 -8477214.346679688,
 -7562018.7431640625,
 -7190127.9423828125,
 -6312803.510986328,
 -5792549.500488281,
 -7880129.286376953,
 -6501093.313720703,
 -5580326.359619141,
 -6729523.923095703,
 -5871317.956787109,
 -7674286.922607422,
 -5876442.632324219]

In [10]:
feasibles

[False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False]

In [7]:
print('Optimal params:', solver.optimal_params)

Optimal params: [ 3.14159308e+00  4.22153095e+00  3.14159312e+00  1.70606676e+00
 -3.95019990e-07  2.54903802e-01  3.14158771e+00  3.14159196e+00
  3.82128988e+00  6.28318468e+00  2.23952066e+00  6.28318529e+00
  5.87626859e+00  4.35982511e+00  3.14159194e+00  2.90519427e+00
  3.14160377e+00  3.14159255e+00  5.20324754e+00  3.14159318e+00
 -1.43552578e+00 -1.76420901e-06  2.54902980e-01  3.14158735e+00
  3.14159287e+00  3.82128971e+00  6.28318586e+00  4.04366441e+00
  3.14150011e+00 -1.47732083e-01  1.18972724e+00  3.14159357e+00
  3.37799151e+00  3.14158088e+00]


In [ ]:
# def obj_func(params):
#     return solver.compute_energy(params)

# result = minimize(obj_func, initial_params, method='cobyla')
# result

In [40]:
qubo_solution = solver.sample_most_likely()
print('QUBO solution:', qubo_solution)

QUBO solution: [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0]


In [41]:
qp_solution = conv.interpret(qubo_solution)
print('QP solution:', qp_solution)

QP solution: [0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 1. 0.]


In [42]:
qp.objective.evaluate(qp_solution)

20.0

In [8]:
counts = solver._sample_optimal_circuit()
counts

TypeError: LCCVQE._sample_optimal_circuit() missing 1 required positional argument: 'shots'

# Solve without LCC

In [ ]:
ansatz = TwoLocal(
    num_qubits=qubo.get_num_vars(),
    rotation_blocks='ry',
    entanglement_blocks='cz',
    entanglement='circular',
    reps=1,
)
result = estimator.run([(ansatz, solver.hamiltonian, initial_params)]).result()[0]
result.data.evs

In [ ]:
def obj_func(params):
    result = estimator.run([(ansatz, solver.hamiltonian, params)]).result()[0]
    return result.data.evs

initial_params = solver.generate_random_params()
result = minimize(obj_func, initial_params, method='l-bfgs-b')
print('Expectation:', -(result.fun + solver.offset))

# Reconstructing the quasi-prob distribution

## Brute force (run the full circuit)

In [ ]:
backend = AerSimulator(method='matrix_product_state')
sampler = BackendSampler(backend=backend)

num_qubits = qp.get_num_vars()
ansatz = TwoLocal(
    num_qubits=num_qubits,
    rotation_blocks='ry',
    entanglement_blocks='cz',
    entanglement='circular',
    reps=1,
)
ansatz.measure_all()

initial_params = np.random.rand(2 * num_qubits) * 2 * np.pi
result = sampler.run(ansatz, initial_params).result()

### Linear entanglement

In [7]:
backend = AerSimulator(method='matrix_product_state')
sampler = BackendSamplerV2(backend=backend)
# sampler = AerSamplerV2()

num_qubits = qp.get_num_vars()
qc = QuantumCircuit(num_qubits)
thetas = ParameterVector('theta', 2 * num_qubits)
for i in range(num_qubits):
    qc.ry(thetas[i], i)

for i in range(num_qubits - 1):
    qc.cx(i, i + 1)

for i in range(num_qubits):
    qc.ry(thetas[num_qubits + i], i)

qc.measure_all()

initial_params = np.random.rand(2 * num_qubits) * 2 * np.pi
result = sampler.run([(qc, initial_params)]).result()[0]

### Full entanglement

In [8]:
backend = AerSimulator(method='matrix_product_state')
sampler = BackendSamplerV2(backend=backend)
# sampler = AerSamplerV2()

num_qubits = qp.get_num_vars()
qc = QuantumCircuit(num_qubits)
thetas = ParameterVector('theta', 2 * num_qubits)
for i in range(num_qubits):
    qc.ry(thetas[i], i)

for i, j in combinations(range(num_qubits), 2):
    qc.cx(i, j)

for i in range(num_qubits):
    qc.ry(thetas[num_qubits + i], i)

qc.measure_all()

initial_params = np.random.rand(2 * num_qubits) * 2 * np.pi
result = sampler.run([(qc, initial_params)]).result()[0]

In [7]:
energy_op, offset = max_cut.to_quadratic_program().to_ising()
energy_op

SparsePauliOp(['IIZIIIIIIZ', 'IZIIIIIIIZ', 'IIIIIIIZZI', 'IZIIIIIIZI', 'IIIIIIZZII', 'ZIIIIIZIII', 'IIIZIZIIII', 'ZIIIIZIIII', 'IIIZZIIIII', 'IIZIZIIIII'],
              coeffs=[0.5+0.j, 0.5+0.j, 0.5+0.j, 0.5+0.j, 0.5+0.j, 0.5+0.j, 0.5+0.j, 0.5+0.j,
 0.5+0.j, 0.5+0.j])

In [9]:
term = energy_op[0]
np.argwhere(term.paulis[0])

Pauli('IIZIIIIIIZ')